# Expand retrieval from the question

[Tutorial 110](./110_hybrid_ranking.ipynb) combined semantic, lexical, and code-aware rankings with ordinary Reciprocal Rank Fusion (RRF).

This tutorial keeps that RRF step unchanged.

QAChat then checks three deterministic repository signals:

1. **Exact file** — an exact repository path or file name is written.
2. **Exact function** — an exact function name is written.
3. **Relationship** — an exact function is present and the question asks
   how that function is connected, called, used, or referenced.

These signals do not change RRF scores. They only guarantee additional
context after the normal RRF top results have been selected.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


## 1. Load the embedding model


In [ ]:
BACKEND = "auto"
ALLOW_DOWNLOAD = True

# Example model for MLX
MODEL_ID = "mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ"

# Example model for PyTorch
# MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"


In [ ]:
from qachat import EmbeddingSession


embedding_session = EmbeddingSession(
    MODEL_ID,
    backend=BACKEND,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")


## 2. Prepare repository data

The three retrieval channels are prepared exactly as before:

- semantic retrieval for meaning;
- lexical retrieval for exact wording;
- code-aware retrieval for repository relationships.


In [ ]:
from qachat import scan_repository


report = scan_repository(
    root,
    extensions=[".py"],
    include_paths=["qachat"],
)


In [ ]:
from qachat import extract_source_units, read_repository_files


source_files = read_repository_files(report)
units = extract_source_units(source_files)

print(f"Source files: {len(source_files)}")
print(f"Natural units: {len(units)}")


In [ ]:
from qachat import chunk_source_units


chunks = chunk_source_units(
    units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=300,
    max_tokens=400,
    overlap_tokens=40,
)

print(f"Searchable chunks: {len(chunks)}")


In [ ]:
from qachat import build_code_graph, build_lexical_index, build_semantic_index


semantic_index = build_semantic_index(chunks, embedding_session)
lexical_index = build_lexical_index(chunks)
code_graph = build_code_graph(source_files, units)


## 3. Four questions isolate the three signals

Q1 has no exact repository target and therefore receives only the
normal RRF results.

Q2 names an exact file.

Q3 names an exact function. 

Q4 names an exact function while also asking about its repository relationships.


In [ ]:
QUESTIONS = {
    "Q1": "What does semantic search mean?",
    "Q2": "Walk me through qachat/retrievals/strategies/retrieval_semantic.py",
    "Q3": "What does semantic_search do?",
    "Q4": "How is semantic_search used across the repository?",
}

for label, question in QUESTIONS.items():
    print(f"{label}: {question}")


## 4. Detect the signals

`plan_retrieval` is deterministic and uses no LLM.
It only records context that should be guaranteed later. It does not
rank chunks and does not change RRF.


In [ ]:
from qachat import plan_retrieval


plans = {
    label: plan_retrieval(question, code_graph)
    for label, question in QUESTIONS.items()
}


In [ ]:
for label, plan in plans.items():
    print(f"\n{'=' * 64}")
    print(f"{label}: {QUESTIONS[label]}")
    print(f"Signals: {', '.join(plan.signals)}")

    paths = [path.as_posix() for path in plan.paths]
    print(
        "Exact files: "
        + (", ".join(paths) if paths else "none")
    )

    print(
        "Exact functions: "
        + (
            ", ".join(plan.functions)
            if plan.functions
            else "none"
        )
    )

    print(
        "Relationship: "
        + ("yes" if plan.relationship else "no")
    )

    print(
        "Distance-1 functions: "
        + (
            ", ".join(plan.related_functions)
            if plan.related_functions
            else "none"
        )
    )


The expected decisions are:

| Question | Deterministic expansion |
| --- | --- |
| Q1 | none |
| Q2 | all chunks from the exact file |
| Q3 | all chunks from the exact function |
| Q4 | exact function + distance-1 related functions |

The important point is that these are guarantees, not alternative rankings.


## 5. Run the three retrieval channels

Each question still goes through semantic, lexical, and code-aware
retrieval. No routing signal changes these searches.


In [ ]:
from qachat import (
    SOFTWARE_REPOSITORY_INSTRUCTION,
    code_search,
    format_qwen3_query,
    lexical_search,
    semantic_search,
)


channel_results = {}

for label, question in QUESTIONS.items():
    semantic_query = format_qwen3_query(
        question=question,
        instruction=SOFTWARE_REPOSITORY_INSTRUCTION,
    )

    channel_results[label] = {
        "semantic": semantic_search(
            semantic_index,
            semantic_query,
            embedding_session,
            count=15,
        ),
        "lexical": lexical_search(
            lexical_index,
            question,
            count=15,
        ),
        "code": code_search(
            code_graph,
            chunks,
            question,
            count=15,
            max_distance=2,
        ),
    }


## 6. Use ordinary RRF

This is the same RRF used in Tutorial 110. There are no routing
weights.

The output here is the general top-N evidence before any deterministic
expansion is added.


In [ ]:
from qachat import hybrid_search


RRF_COUNT = 5
rrf_results = {}

for label, results in channel_results.items():
    rrf_results[label] = hybrid_search(
        semantic=results["semantic"],
        lexical=results["lexical"],
        code=results["code"],
        count=RRF_COUNT,
    )


In [ ]:
for label, results in rrf_results.items():
    print(f"\n{'=' * 64}")
    print(f"{label}: {QUESTIONS[label]}")
    print(f"Normal RRF top {RRF_COUNT}:")

    for rank, result in enumerate(results, start=1):
        name = result.chunk.symbol or result.chunk.path.name
        print(
            f"{rank}. {result.score:.4f} "
            f"{result.chunk.path}::{name}"
        )


## 7. Expand the RRF evidence

`expand_retrieval_context` keeps the normal RRF chunks first.

It then appends any guaranteed context requested by the deterministic
signals:

- every chunk from an exact file;
- every chunk from an exact function;
- every chunk from distance-1 related functions.

Duplicate chunks are removed.


In [ ]:
from qachat import expand_retrieval_context

expanded_results = {
    label: expand_retrieval_context(
        rrf_results[label],
        chunks,
        code_graph,
        plans[label],
    )
    for label in QUESTIONS
}


## 8. Show exactly what was added

This intermediate result separates the original RRF evidence from the
guaranteed expansion. That makes the behavior easy to verify.


In [ ]:
for label, expanded in expanded_results.items():
    rrf_ids = {
        result.chunk.identifier
        for result in rrf_results[label]
    }
    added = [
        chunk
        for chunk in expanded
        if chunk.identifier not in rrf_ids
    ]

    print(f"\n{'=' * 64}")
    print(f"{label}: {QUESTIONS[label]}")
    print(f"Signals: {', '.join(plans[label].signals)}")
    print(f"RRF chunks: {len(rrf_results[label])}")
    print(f"Added chunks: {len(added)}")
    print(f"Final evidence chunks: {len(expanded)}")

    if not added:
        print("Added context: none")
        continue

    print("Added context:")
    for chunk in added:
        name = chunk.symbol or chunk.path.name
        print(
            f"  {chunk.path}::{name} "
            f"lines {chunk.start_line}-{chunk.end_line}"
        )


## 9. Verify the exact-file expansion

For Q2, every searchable chunk from the requested file should be
present in the final evidence, even if it was not in the RRF top five.


In [ ]:
q2_plan = plans["Q2"]
q2_paths = set(q2_plan.paths)

expected_q2 = {
    chunk.identifier
    for chunk in chunks
    if chunk.path in q2_paths
}
actual_q2 = {
    chunk.identifier
    for chunk in expanded_results["Q2"]
}

print(f"Requested file chunks: {len(expected_q2)}")
print(
    "All requested file chunks included: "
    f"{expected_q2 <= actual_q2}"
)


## 10. Verify the exact-function expansion

For Q3, every chunk belonging to `semantic_search` should be included.
If the function fits in one natural chunk, this will normally be one
complete function chunk.


In [ ]:
q3_plan = plans["Q3"]
q3_names = {
    name.rsplit(".", 1)[-1]
    for name in q3_plan.functions
}

q3_function_chunks = [
    chunk
    for chunk in expanded_results["Q3"]
    if chunk.symbol in q3_names
]

print(f"Matched functions: {q3_plan.functions}")
print(f"Function chunks included: {len(q3_function_chunks)}")

for chunk in q3_function_chunks:
    print(
        f"{chunk.path} "
        f"lines {chunk.start_line}-{chunk.end_line}"
    )
    print()
    print(chunk.text)


## 11. Verify the relationship expansion

For Q4, the exact function is kept and direct function neighbors are
added from both directions of the code graph.

For a `calls` relationship this means direct callers and direct
callees. Other direct function-to-function relationships are also
eligible when they exist in the graph.


In [ ]:
q4_plan = plans["Q4"]
targets = set(q4_plan.functions)

print(f"Target functions: {q4_plan.functions}")
print(f"Distance-1 functions: {q4_plan.related_functions}")
print()

for relation in code_graph.relations:
    touches_target = (
        relation.source in targets
        or relation.target in targets
    )
    if touches_target:
        print(
            f"{relation.source} "
            f"--{relation.relation}--> "
            f"{relation.target}"
        )


## 12. Inspect the final evidence

The final evidence begins with the normal RRF top results and then
contains any additional guaranteed context.

This is the context that can later be supplied to the LLM.


In [ ]:
for label, expanded in expanded_results.items():
    print(f"\n{'=' * 64}")
    print(f"{label}: {QUESTIONS[label]}")
    print(f"Final evidence chunks: {len(expanded)}")

    for rank, chunk in enumerate(expanded, start=1):
        name = chunk.symbol or chunk.path.name
        print(
            f"{rank}. {chunk.path}::{name} "
            f"lines {chunk.start_line}-{chunk.end_line}"
        )


## 13. Release the model


In [ ]:
embedding_session.close()

print("Embedding model released.")
